In [ ]:
import pymysql
import pandas as pd
import ollama
import random
import time

# Database connections
read_conn = pymysql.connect(
    host='mysql.clarksonmsda.org',
    port=3306,
    user='ia626',
    passwd='ia626clarkson',
    db='ia626',
    cursorclass=pymysql.cursors.DictCursor
)

write_conn = pymysql.connect(
    host='mysql.clarksonmsda.org',
    port=3306,
    user='ia626',
    passwd='ia626clarkson',
    db='ia626',
    cursorclass=pymysql.cursors.DictCursor
)

In [ ]:
# function to generate review and random rating
def generate_review_and_rating(title, author, model='llama3'):
    prompt = f"""
Write a concise and informative review (around 100-150 words) for the book "{title}" by {author}.
Only output the review text directly, without any introductions or explanations.
"""
    response = ollama.chat(model=model, messages=[{'role': 'user', 'content': prompt}])
    review = response['message']['content'].strip()

    rating = random.randint(2, 5)
    return review, rating

In [ ]:
# function to generate review summary
def generate_review_summary(title, author, model='llama3'):
    prompt = f"""Create a short, catchy review headline (less than 10 words) for the book "{title}" by {author}.
Only return the headline, no extra text."""
    response = ollama.chat(model=model, messages=[{'role': 'user', 'content': prompt}])
    summary = response['message']['content'].strip()
    return summary

In [ ]:
# function to get books from books table
def fetch_books():
    with read_conn.cursor() as cursor:
        sql = "SELECT book_id, title, author FROM books"  # Update table/column names
        cursor.execute(sql)
        books = cursor.fetchall()
    return books

In [ ]:
# function to insert generated reviews into reviews table
def insert_review(book_id, rating, summary, review):
    with write_conn.cursor() as cursor:
        sql = """
        INSERT INTO reviews (book_id, rating, review_summary, review_text, source)
        VALUES (%s, %s, %s, %s, %s)
        """
        cursor.execute(sql, (book_id, rating, summary, review,  'AI'))
    write_conn.commit()

In [ ]:
# pathway to generate and insert reviews
def generate_and_insert_reviews():
    books = fetch_books()
    for book in books:
        try:
            # generate review and summary
            summary = generate_review_summary(book['title'], book['author'])
            review, rating = generate_review_and_rating(book['title'], book['author'])

            # Insert into DB
            insert_review(book['book_id'], rating, summary, review)
            print(f"Inserted AI review for: {book['title']} by {book['author']}")
            time.sleep(1)  
        except Exception as e:
            print(f"Failed on {book['title']} by {book['author']}: {e}")

if __name__ == "__main__":
    generate_and_insert_reviews()